# Notebook 03: Pinecone Vector Database with LangChain & LangSmith

## Overview
Create and populate Pinecone vector database using **LangChain** with **LangSmith** monitoring.

## Key Features:
- ✅ **LangChain Pinecone integration** 
- ✅ **LangSmith tracing** 
- ✅ OpenAI embeddings via LangChain
- Rich metadata for RAG retrieval
- Batch processing with progress tracking
- Retrieval validation tests

## Output:
- Populated Pinecone index
- LangSmith traces for monitoring
- Ingestion manifest

## 1. Imports

In [2]:
# Run if needed
# !pip install langchain langchain-openai langchain-pinecone pinecone-client python-dotenv langsmith pandas tqdm

In [1]:
import os
import json
import glob
from datetime import datetime
from typing import List, Dict, Any

import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv


# LangChain imports 
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain.docstore.document import Document

from pinecone import Pinecone, ServerlessSpec

import numpy as np
import matplotlib.pyplot as plt

from langsmith import Client

## 2. Configuration

Adjust these values before running the notebook.

### Notes
- `PROCESSED_DIR` should point to your processed chunk JSON files.
- `CHUNK_GLOB` is set for the timestamped chunk files created earlier.
- `INDEX_NAME` should be lowercase and use only allowed Pinecone characters.
- `NAMESPACE` lets you isolate this corpus inside the same index.
- `EMBEDDING_MODEL` is set to `text-embedding-3-small` by default.

## 3. Load API keys

This notebook expects:

- `OPENAI_API_KEY`
- `PINECONE_API_KEY`
- `LANGCHAIN_API_KEY`(`LANGSMITH_API_KEY`)

You can store them in a `.env` file or set them as environment variables.

In [2]:
# Load environment variables
load_dotenv()

# API Keys
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")

# LangSmith Configuration
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "youtube-rag-engineering-chatbot"
os.environ["LANGCHAIN_API_KEY"] = LANGSMITH_API_KEY or ""

# Pinecone Configuration
INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "youtube-rag-mechanical-engineering")
NAMESPACE = os.getenv("PINECONE_NAMESPACE", "efficient-engineer-v3")
PINECONE_CLOUD = os.getenv("PINECONE_CLOUD", "aws")
PINECONE_REGION = os.getenv("PINECONE_REGION", "us-east-1")

# Model Configuration
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
EMBEDDING_DIMENSIONS = 1536

# Processing Configuration
BATCH_SIZE = 100

# Data Paths
PROCESSED_DIR = "../data/processed"
MANIFEST_FILE = os.path.join(PROCESSED_DIR, "pinecone_ingestion_manifest.json")

# Validate keys
assert OPENAI_API_KEY, "❌ OPENAI_API_KEY not set"
assert PINECONE_API_KEY, "❌ PINECONE_API_KEY not set"
assert LANGSMITH_API_KEY, "❌ LANGSMITH_API_KEY not set (required by Carlos)"

print("✅ Configuration loaded")
print(f"   Index: {INDEX_NAME}")
print(f"   Namespace: {NAMESPACE}")
print(f"   Embedding Model: {EMBEDDING_MODEL}")
print(f"   LangSmith Project: {os.environ['LANGCHAIN_PROJECT']}")
print(f"   LangSmith Tracing: {os.environ['LANGCHAIN_TRACING_V2']}")

✅ Configuration loaded
   Index: youtube-rag-mechanical-engineering
   Namespace: efficient-engineer-v3
   Embedding Model: text-embedding-3-small
   LangSmith Project: youtube-rag-engineering-chatbot
   LangSmith Tracing: true


## 4. Initialize LangChain Components

In [3]:
# Initialize OpenAI Embeddings via LangChain
embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    openai_api_key=OPENAI_API_KEY,
)

# Initialize LangSmith client for monitoring
langsmith_client = Client(api_key=LANGSMITH_API_KEY)

print("✅ LangChain OpenAI Embeddings initialized")
print("✅ LangSmith client initialized")

# Test embedding (with LangSmith tracing)
test_text = "This is a test embedding"
test_embedding = embeddings.embed_query(test_text)
print(f"✅ Test embedding successful (dimension: {len(test_embedding)})")
print(f"   Check LangSmith dashboard for traces!")

✅ LangChain OpenAI Embeddings initialized
✅ LangSmith client initialized
✅ Test embedding successful (dimension: 1536)
   Check LangSmith dashboard for traces!


## 5. Initialize Pinecone

In [4]:
# Initialize Pinecone client
pc = Pinecone(api_key=PINECONE_API_KEY)

# Check if index exists
existing_indexes = [idx.name for idx in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    print(f"📝 Creating new index: {INDEX_NAME}")
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIMENSIONS,
        metric="cosine",
        spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
    )
    print("✅ Index created")
else:
    print(f"✅ Index '{INDEX_NAME}' already exists")

# Get index
index = pc.Index(INDEX_NAME)

# Check stats
stats = index.describe_index_stats()
print(f"\n📊 Index Statistics:")
print(f"   Total vectors: {stats.total_vector_count}")
print(f"   Dimension: {stats.dimension}")
print(f"   Namespaces: {list(stats.namespaces.keys()) if stats.namespaces else []}")

✅ Index 'youtube-rag-mechanical-engineering' already exists

📊 Index Statistics:
   Total vectors: 1770
   Dimension: 1536
   Namespaces: ['efficient-engineer-v1', 'efficient-engineer-v3']


## 6. Discover all processed chunk files

In [5]:
# Find all processed chunk files
chunk_files = sorted(glob.glob(os.path.join(PROCESSED_DIR, "chunks_*.json")))

print(f"📁 Found {len(chunk_files)} processed chunk files")

# Load all chunks
all_chunks = []
for chunk_file in tqdm(chunk_files, desc="Loading chunks"):
    with open(chunk_file, "r", encoding="utf-8") as f:
        chunks = json.load(f)
        all_chunks.extend(chunks)

print(f"✅ Loaded {len(all_chunks):,} chunks total")

if not all_chunks:
    raise ValueError("No chunks found! Run Notebook 02 first.")

📁 Found 65 processed chunk files


Loading chunks:   0%|          | 0/65 [00:00<?, ?it/s]

✅ Loaded 1,475 chunks total


## 7. Convert Chunks to LangChain Documents

In [21]:
def chunk_to_langchain_document(chunk: Dict[str, Any]) -> Document:
    """Convert chunk dict to LangChain Document with metadata."""
    
    # Extract metadata
    chunk_meta = chunk.get("metadata", {})
    
    # Build comprehensive metadata for Pinecone
    metadata = {
        # Core identifiers
        "chunk_id": chunk["chunk_id"],
        "video_id": chunk["video_id"],
        "chunk_index": chunk["chunk_index"],
        "total_chunks": chunk.get("total_chunks", 0),
        
        # Video info
        "video_title": chunk_meta.get("video_title", ""),
        "video_url": chunk_meta.get("video_url", ""),
        "channel": chunk_meta.get("channel", ""),
        
        # Timestamps
        "start_time": chunk["start_time"],
        "end_time": chunk["end_time"],
        "start_time_str": chunk["start_time_str"],
        "end_time_str": chunk["end_time_str"],
        "duration_seconds": chunk.get("duration_seconds", 0),
        
        # Text stats
        "char_count": chunk["char_count"],
        "word_count": chunk["word_count"],
        
        # Additional metadata
        "upload_date": chunk_meta.get("upload_date", ""),
        "transcript_language": chunk_meta.get("transcript_language", "en"),
        
        # For filtering
        "tags": ",".join(chunk_meta.get("tags", [])[:5]),  # First 5 tags
        "categories": ",".join(chunk_meta.get("categories", [])[:3]),
    }
    
    # Create Document
    return Document(
        page_content=chunk["text"],
        metadata=metadata,
    )


# Convert all chunks to LangChain Documents
print("\n🔄 Converting chunks to LangChain Documents...")
documents = [chunk_to_langchain_document(chunk) for chunk in tqdm(all_chunks, desc="Converting")]

print(f"✅ Created {len(documents):,} LangChain Documents")
print(f"\nSample document metadata:")
print(json.dumps(documents[0].metadata, indent=2))


🔄 Converting chunks to LangChain Documents...


Converting:   0%|          | 0/1475 [00:00<?, ?it/s]

✅ Created 1,475 LangChain Documents

Sample document metadata:
{
  "chunk_id": "-CGxZ-qn4HA_chunk_0000",
  "video_id": "-CGxZ-qn4HA",
  "chunk_index": 0,
  "total_chunks": 25,
  "video_title": "Weighted integral & Weak formulation",
  "video_url": "https://www.youtube.com/watch?v=-CGxZ-qn4HA",
  "channel": "Basics of Finite Element Analysis-I",
  "start_time": 13.7,
  "end_time": 82.68,
  "start_time_str": "00:13",
  "end_time_str": "01:22",
  "duration_seconds": 68.98,
  "char_count": 533,
  "word_count": 94,
  "upload_date": "20160201",
  "transcript_language": "en",
  "tags": "Weighted,integral,weak,formulation",
  "categories": "People & Blogs"
}


## 8. Create Pinecone VectorStore using LangChain

In [22]:
print("\n🚀 Creating Pinecone VectorStore via LangChain...")
print(f"   This will embed {len(documents):,} documents and upload to Pinecone")
print(f"   With batch size {BATCH_SIZE}, this may take a few minutes...")
print(f"   ⚡ LangSmith is tracing all operations!\n")

# Create VectorStore
vectorstore = PineconeVectorStore.from_documents(
    documents=documents,
    embedding=embeddings,
    index_name=INDEX_NAME,
    namespace=NAMESPACE,
)

print("\n✅ Pinecone VectorStore created successfully!")
print(f"   Index: {INDEX_NAME}")
print(f"   Namespace: {NAMESPACE}")
print(f"   Documents: {len(documents):,}")
print(f"\n📊 Check LangSmith dashboard for embedding traces!")


🚀 Creating Pinecone VectorStore via LangChain...
   This will embed 1,475 documents and upload to Pinecone
   With batch size 100, this may take a few minutes...
   ⚡ LangSmith is tracing all operations!


✅ Pinecone VectorStore created successfully!
   Index: youtube-rag-mechanical-engineering
   Namespace: efficient-engineer-v3
   Documents: 1,475

📊 Check LangSmith dashboard for embedding traces!


## 9. Test Retrieval with LangChain

In [23]:
print("\n🔍 Testing retrieval via LangChain VectorStore...\n")

# Test query
test_query = "What is stress and strain in engineering?"

# Use LangChain similarity search (automatically traced by LangSmith)
results = vectorstore.similarity_search(
    query=test_query,
    k=3,
    namespace=NAMESPACE,
)

print(f"Query: '{test_query}'\n")
print(f"Found {len(results)} results:\n")

for i, doc in enumerate(results, 1):
    meta = doc.metadata
    print(f"Result {i}:")
    print(f"   Title: {meta.get('video_title', 'N/A')}")
    print(f"   Timestamp: {meta.get('start_time_str')} -> {meta.get('end_time_str')}")
    print(f"   Channel: {meta.get('channel', 'N/A')}")
    print(f"   Text preview: {doc.page_content[:200]}...")
    print()

print("✅ Retrieval test successful!")
print("📊 Check LangSmith for retrieval trace!")


🔍 Testing retrieval via LangChain VectorStore...

Query: 'What is stress and strain in engineering?'

Found 3 results:

Result 1:
   Title: Understanding True Stress and True Strain
   Timestamp: 03:56 -> 04:38
   Channel: The Efficient Engineer
   Text preview: . The definition of engineering strain is that it is the change in length divided by the original length. We can re-arrange this equation to the form, L divided by L0, minus one. And we can use this t...

Result 2:
   Title: Understanding True Stress and True Strain
   Timestamp: 00:32 -> 01:26
   Channel: The Efficient Engineer
   Text preview: . These stress and strain values are actually just approximations of the true stress and strain in the specimen. We call them engineering stress and engineering strain, and denote them using the subsc...

Result 3:
   Title: Understanding True Stress and True Strain
   Timestamp: 02:32 -> 03:18
   Channel: The Efficient Engineer
   Text preview: . Examples of this might be the analysis

## 10. Test Retrieval with Scores

In [24]:
# Test with similarity scores
results_with_scores = vectorstore.similarity_search_with_score(
    query=test_query,
    k=5,
    namespace=NAMESPACE,
)

print(f"\n📊 Top {len(results_with_scores)} results with scores:\n")

retrieval_data = []
for i, (doc, score) in enumerate(results_with_scores, 1):
    meta = doc.metadata
    retrieval_data.append({
        "rank": i,
        "score": round(score, 4),
        "video_title": meta.get("video_title", "N/A")[:50],
        "timestamp": f"{meta.get('start_time_str')} -> {meta.get('end_time_str')}",
        "channel": meta.get("channel", "N/A"),
    })

df_retrieval = pd.DataFrame(retrieval_data)
display(df_retrieval)


📊 Top 5 results with scores:



,rank,score,video_title,timestamp,channel
0,1,0.7155,Understanding True Stress and True Strain,03:56 -> 04:38,The Efficient Engineer
1,2,0.6967,Understanding True Stress and True Strain,00:32 -> 01:26,The Efficient Engineer
2,3,0.6510,Understanding True Stress and True Strain,02:32 -> 03:18,The Efficient Engineer
3,4,0.6297,Understanding True Stress and True Strain,04:54 -> 05:24,The Efficient Engineer
4,5,0.6257,An Introduction to Stress and Strain,00:00 -> 00:52,The Efficient Engineer


## 11. Run Retrieval Validation Suite

In [25]:
# Define test queries (customize for your domain)
test_cases = [
    {
        "query": "Explain stress and strain",
        "expected_keywords": ["stress", "strain", "deformation"],
    },
    {
        "query": "What is Young's modulus?",
        "expected_keywords": ["young", "modulus", "elastic"],
    },
    {
        "query": "How does material fail under load?",
        "expected_keywords": ["fail", "load", "material"],
    },
]

print("\n🧪 Running retrieval validation suite...\n")

validation_results = []

for test in test_cases:
    query = test["query"]
    keywords = test["expected_keywords"]
    
    # Search
    results = vectorstore.similarity_search(query, k=3, namespace=NAMESPACE)
    
    # Check if keywords appear in results
    all_text = " ".join([doc.page_content.lower() for doc in results])
    keywords_found = [kw for kw in keywords if kw.lower() in all_text]
    
    validation_results.append({
        "query": query,
        "expected_keywords": ", ".join(keywords),
        "keywords_found": ", ".join(keywords_found),
        "pass": len(keywords_found) > 0,
        "top_score": "N/A",  # Can add if using similarity_search_with_score
    })

df_validation = pd.DataFrame(validation_results)
print("📊 Validation Results:")
display(df_validation)

pass_rate = df_validation["pass"].mean()
print(f"\n✅ Pass rate: {pass_rate:.1%}")

if pass_rate >= 0.8:
    print("✅ Retrieval quality: EXCELLENT")
elif pass_rate >= 0.6:
    print("🟢 Retrieval quality: GOOD")
else:
    print("⚠️  Retrieval quality: NEEDS IMPROVEMENT")


🧪 Running retrieval validation suite...

📊 Validation Results:


,query,expected_keywords,keywords_found,pass,top_score
0,Explain stress and strain,"stress, strain, deformation","stress, strain, deformation",True,N/A
1,What is Young's modulus?,"young, modulus, elastic","young, modulus, elastic",True,N/A
2,How does material fail under load?,"fail, load, material","fail, load, material",True,N/A



✅ Pass rate: 100.0%
✅ Retrieval quality: EXCELLENT


## 12. Save Ingestion Manifest

In [26]:
# Create ingestion manifest
manifest = {
    "ingestion_timestamp": datetime.now().isoformat(),
    "index_name": INDEX_NAME,
    "namespace": NAMESPACE,
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dimensions": EMBEDDING_DIMENSIONS,
    "total_documents": len(documents),
    "total_videos": len(set([doc.metadata["video_id"] for doc in documents])),
    "langchain_version": "Used LangChain Pinecone integration",
    "langsmith_project": os.environ["LANGCHAIN_PROJECT"],
    "langsmith_tracing": os.environ["LANGCHAIN_TRACING_V2"],
    "validation_pass_rate": f"{pass_rate:.1%}",
}

# Save manifest
with open(MANIFEST_FILE, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(f"\n📄 Ingestion manifest saved to: {MANIFEST_FILE}")
print(json.dumps(manifest, indent=2))


📄 Ingestion manifest saved to: ../data/processed\pinecone_ingestion_manifest.json
{
  "ingestion_timestamp": "2026-03-09T14:07:53.970898",
  "index_name": "youtube-rag-mechanical-engineering",
  "namespace": "efficient-engineer-v3",
  "embedding_model": "text-embedding-3-small",
  "embedding_dimensions": 1536,
  "total_documents": 1475,
  "total_videos": 65,
  "langchain_version": "Used LangChain Pinecone integration",
  "langsmith_project": "youtube-rag-engineering-chatbot",
  "langsmith_tracing": "true",
  "validation_pass_rate": "100.0%"
}
